# Create MMVEC tbls of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [1028]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [1029]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [1030]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [1031]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [1032]:
skin_group = "Acne_L"
# skin_group = "Acne_NL"
# skin_group = "Healthy"

In [1033]:
# Read in metabolomics table and set index
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/data_sample_10282024.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in VIP-filtered feature list
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_with_shortened_names.tsv', sep='\t')

# Get list of VIP feature IDs
vip_features = VIP_filtered['ID'].astype(str).unique().tolist()

# Extract the feature ID from each column name (assumes feature ID is before the first semicolon)
col_ids = [col.split(';')[0] for col in metaB_tbl.columns]

# Map original column names to their extracted feature IDs
col_map = dict(zip(metaB_tbl.columns, col_ids))

# Keep only columns where the extracted ID matches a VIP feature
cols_to_keep = [col for col, cid in col_map.items() if cid in vip_features]

# Subset the table to keep only those columns
metaB_tbl = metaB_tbl[cols_to_keep]

# Rename columns using Shortened_Compound_Name from VIP_filtered
# Create mapping of ID to Shortened_Compound_Name
name_map = dict(zip(VIP_filtered['ID'].astype(str), VIP_filtered['Shortened_Compound_Name']))

# Create a mapping from feature ID to shortened name, mz, and RT
VIP_filtered['ID'] = VIP_filtered['ID'].astype(str)  # Ensure IDs are strings
name_map = VIP_filtered.set_index('ID')['Shortened_Compound_Name'].to_dict()
mz_map = VIP_filtered.set_index('ID')['mz'].to_dict()
rt_map = VIP_filtered.set_index('ID')['RT'].to_dict()

# Build new column names
new_cols = []
for col in metaB_tbl.columns:
    feature_id = col.split(';')[0]  # Extract feature ID
    short_name = name_map.get(feature_id, feature_id)
    mz = mz_map.get(feature_id, 'NA')
    rt = rt_map.get(feature_id, 'NA')
    new_col_name = f"{short_name} (mz: {mz}, RT: {rt})"
    new_cols.append(new_col_name)

# Rename the columns
metaB_tbl.columns = new_cols

# View the result
metaB_tbl


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.838,0.0000,0.000,0.000,0.00,0.0000,125692.3300
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.390,0.0000,0.000,0.000,0.00,0.0000,10553.9270
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.00,750870.100,1618142.800,442653.660,286004.00,...,87817.2000,5348.9400,231525.840,14299.352,0.0000,0.000,0.000,0.00,3822.6377,4734.5693
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.309,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.043,0.0000,0.000,0.000,0.00,0.0000,0.0000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.117,0.0000,0.000,0.000,0.00,0.0000,220846.1600
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.246,0.0000,0.000,0.000,0.00,0.0000,25695.0140


In [1034]:
# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_uncollapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)
# Extract the feature name after the last semicolon for each column
# V1V3_tbl.columns = V1V3_tbl.columns.str.split(';').str[-1].str.strip()
# # Sort columns by their sum in descending order
# V1V3_tbl = V1V3_tbl.loc[:, V1V3_tbl.sum().sort_values(ascending=False).index]


# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_uncollapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [1035]:
V1V3_tbl

,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Propionibacteriales; f__Propionibacteriaceae; g__Cutibacterium|ASV1,d__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Neisseriaceae; g__uncultured|ASV3,d__Bacteria; p__Firmicutes; c__Bacilli; o__Staphylococcales; f__Staphylococcaceae; g__Staphylococcus|ASV8,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Corynebacteriales; f__Corynebacteriaceae; g__Lawsonella,d__Bacteria; p__Firmicutes; c__Bacilli; o__Lactobacillales; f__Streptococcaceae; g__Streptococcus|ASV11,d__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Neisseriaceae; g__uncultured|ASV1,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Propionibacteriales; f__Propionibacteriaceae; g__Cutibacterium|ASV4,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Corynebacteriales; f__Corynebacteriaceae; g__Corynebacterium|ASV13,d__Bacteria; p__Firmicutes; c__Bacilli; o__Lactobacillales; f__Streptococcaceae; g__Streptococcus|ASV4,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Micrococcales; f__Micrococcaceae; g__Micrococcus,...,d__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Comamonadaceae|ASV2,d__Bacteria; p__Acidobacteriota; c__Vicinamibacteria; o__Vicinamibacterales; f__Vicinamibacteraceae; g__Luteitalea,d__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Burkholderiales; f__Methylophilaceae; g__Methylotenera,d__Bacteria; p__Actinobacteriota; c__Actinobacteria; o__Micrococcales; f__Microbacteriaceae,d__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Rhizobiales; f__Rhizobiaceae; g__Pseudochrobactrum,d__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Sphingomonadales; f__Sphingomonadaceae; g__Sphingopyxis|ASV2,d__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Caulobacterales; f__Caulobacteraceae; g__Phenylobacterium|ASV3,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Fusicatenibacter,d__Bacteria; p__Firmicutes; c__Clostridia; o__Oscillospirales; f__Ruminococcaceae; g__Ruminococcus,d__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Caulobacterales; f__Caulobacteraceae; g__Phenylobacterium|ASV5
LAMI.RD001.D0.C1,2355.0,0.0,83.0,91.0,77.0,0.0,0.0,10.0,85.0,37.0,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,1781.0,0.0,155.0,120.0,89.0,0.0,27.0,34.0,98.0,52.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C1,2173.0,0.0,81.0,253.0,108.0,0.0,44.0,27.0,12.0,30.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C2,1738.0,0.0,137.0,151.0,225.0,0.0,20.0,44.0,39.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D28.C1,2352.0,3.0,115.0,217.0,162.0,0.0,11.0,30.0,43.0,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,1888.0,1422.0,8.0,8.0,0.0,424.0,11.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C3,988.0,2709.0,8.0,9.0,1.0,31.0,13.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2224.0,1410.0,8.0,27.0,0.0,0.0,9.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D9.C1,1219.0,2006.0,13.0,31.0,1.0,484.0,11.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1036]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.000,125692.330
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.3900,0.0000,0.000,0.000,0.00,0.000,10553.927
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.627,12473.092
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.586,40673.110
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.00,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.000,49464.902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.000,0.000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.000,220846.160
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.000,25695.014
LAMI.RD319.D2.C3,171846.72,106901.95,0.0000,1407.5230,25369.129,0.00,59064.336,833009.000,23485.404,2838344.20,...,25342.2100,0.0000,55986.290,0.0000,0.0000,0.000,0.000,0.00,0.000,11039.730


In [1037]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,0.265648,0.180123,0.002630,0.003544,0.002515,0.000000,0.086299,0.172743,0.050465,0.130990,...,0.005514,0.001103,0.054395,0.001492,0.000000,0.000000,0.000000,0.000000,0.000000,0.011177
LAMI.RD306.D9.C2,0.177092,0.195092,0.002975,0.000000,0.021844,0.015442,0.092514,0.308376,0.058006,0.026585,...,0.013238,0.001764,0.037799,0.001110,0.000000,0.000000,0.000000,0.000000,0.000000,0.000911
LAMI.RD304.D11.C1,0.269227,0.212329,0.007015,0.005441,0.002279,0.000000,0.088183,0.237725,0.048588,0.044292,...,0.008675,0.001392,0.011622,0.000000,0.000796,0.001662,0.003486,0.014728,0.004416,0.001847
LAMI.RD304.D11.C2,0.274761,0.172246,0.000844,0.000000,0.003662,0.000000,0.074110,0.211162,0.037169,0.046760,...,0.004916,0.000558,0.063903,0.003849,0.000656,0.003995,0.008503,0.038416,0.010752,0.009920
LAMI.RD304.D0.C1,0.172611,0.162410,0.000949,0.000698,0.009426,0.000000,0.061201,0.275662,0.028620,0.147184,...,0.009428,0.000000,0.089115,0.000673,0.000716,0.000000,0.000000,0.000000,0.000000,0.010318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.210388,0.224149,0.001544,0.001755,0.002638,0.000000,0.103823,0.247600,0.089449,0.042334,...,0.004404,0.000922,0.022046,0.010158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD308.D9.C3,0.139416,0.174247,0.000802,0.000780,0.004545,0.000000,0.071444,0.188158,0.066626,0.076921,...,0.006204,0.000000,0.016641,0.005532,0.000000,0.000000,0.000000,0.000000,0.000000,0.055971
LAMI.RD319.D2.C2,0.071895,0.064285,0.000478,0.000000,0.001260,0.000000,0.030511,0.142568,0.014543,0.645566,...,0.004206,0.000000,0.008744,0.002357,0.000000,0.000000,0.000000,0.000000,0.000000,0.003068
LAMI.RD319.D2.C3,0.040939,0.025467,0.000000,0.000335,0.006044,0.000000,0.014071,0.198447,0.005595,0.676175,...,0.006037,0.000000,0.013338,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002630


In [1038]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V1V3_tbl_relative.index = metaB_V1V3_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V1V3_tbl_relative.index)]

# Subset the DataFrame
metaB_V1V3_tbl_relative = metaB_V1V3_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V1V3_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20663/3342871550.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD310.D21.C1,0.161275,0.127148,0.003527,0.023439,0.003459,0.141652,0.055794,0.284922,0.031579,0.118801,...,0.005883,0.000000,0.002835,0.008138,0.000000,0.0,0.0,0.000000,0.0,0.017880
LAMI.RD306.D18.C2,0.215800,0.170360,0.000851,0.002343,0.007131,0.001499,0.076037,0.219199,0.041103,0.185253,...,0.011764,0.001530,0.019971,0.002833,0.000430,0.0,0.0,0.008514,0.0,0.007811
LAMI.RD306.D7.C2,0.234776,0.210658,0.008428,0.004283,0.003240,0.001876,0.089672,0.267062,0.048207,0.057453,...,0.008393,0.000511,0.009765,0.000453,0.000635,0.0,0.0,0.011246,0.0,0.000000
LAMI.RD317.D14.C2,0.208521,0.147695,0.001010,0.001145,0.001750,0.019631,0.066396,0.136326,0.038252,0.280606,...,0.006773,0.001677,0.049383,0.003272,0.001062,0.0,0.0,0.000493,0.0,0.000000
LAMI.RD305.D23.C1,0.190355,0.141207,0.001642,0.000000,0.003346,0.000000,0.053749,0.187478,0.029649,0.223439,...,0.007481,0.000144,0.123228,0.003513,0.000000,0.0,0.0,0.000000,0.0,0.012840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD305.D28.C1,0.140246,0.131700,0.002039,0.000000,0.004383,0.000000,0.048748,0.233223,0.026580,0.294372,...,0.012756,0.000000,0.083917,0.002572,0.000000,0.0,0.0,0.000000,0.0,0.001650
LAMI.RD313.D4.C1,0.234309,0.184645,0.001594,0.002245,0.011487,0.000000,0.088735,0.307286,0.044712,0.059806,...,0.004936,0.000000,0.012576,0.010067,0.000891,0.0,0.0,0.000000,0.0,0.010636
LAMI.RD305.D0.C2,0.167853,0.102316,0.000621,0.003366,0.009479,0.000000,0.047204,0.283517,0.025330,0.281149,...,0.009113,0.000000,0.040475,0.002807,0.000000,0.0,0.0,0.000000,0.0,0.000000
LAMI.RD317.D21.C1,0.098352,0.080178,0.000415,0.000000,0.001394,0.015098,0.034518,0.100826,0.023286,0.576273,...,0.005320,0.000883,0.027064,0.002630,0.000000,0.0,0.0,0.006146,0.0,0.006610


In [1039]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V1V3_tbl_relative, output_path)

In [1040]:
# Subset rows where sample is in the metabolomics table
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl_relative.index)]

# Clean column names to just genus|ASV (strip everything before last ;)
V1V3_tbl_matched.columns = V1V3_tbl_matched.columns.str.split(';').str[-1].str.strip()

# Add |ASV1 to column names that don't have |
V1V3_tbl_matched.columns = [col if '|' in col else col + '|ASV1' for col in V1V3_tbl_matched.columns]

# List of genus prefixes
top_V1V3_features = ['g__Cutibacterium', 'g__Staphylococcus', 'g__Streptococcus',
                     'g__Lawsonella', 'g__Micrococcus', 'g__Lactobacillus',
                     'g__Rothia', 'g__Kocuria', 'g__Haemophilus', 
                     'g__Corynebacterium', 'g__Veillonella']

# Keep only columns that start with one of the genus names
matching_cols = [col for col in V1V3_tbl_matched.columns if col.startswith(tuple(top_V1V3_features))]

# Subset the dataframe
V1V3_tbl_matched = V1V3_tbl_matched[matching_cols]


V1V3_tbl_matched

,g__Cutibacterium|ASV1,g__Staphylococcus|ASV8,g__Lawsonella|ASV1,g__Streptococcus|ASV11,g__Cutibacterium|ASV4,g__Corynebacterium|ASV13,g__Streptococcus|ASV4,g__Micrococcus|ASV1,g__Veillonella|ASV3,g__Streptococcus|ASV6,...,g__Veillonella|ASV1,g__Staphylococcus|ASV4,g__Lactobacillus|ASV2,g__Streptococcus|ASV7,g__Corynebacterium|ASV8,g__Streptococcus|ASV1,g__Staphylococcus|ASV3,g__Corynebacterium|ASV4,g__Staphylococcus|ASV7,g__Staphylococcus|ASV2
LAMI.RD304.D14.C2,3380.0,177.0,23.0,40.0,19.0,25.0,15.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
LAMI.RD305.D14.C1,2900.0,30.0,7.0,5.0,0.0,31.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D4.C2,3058.0,535.0,3.0,1.0,47.0,25.0,0.0,8.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D16.C1,3414.0,122.0,8.0,5.0,1.0,101.0,4.0,1.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
LAMI.RD304.D7.C1,3373.0,228.0,14.0,4.0,2.0,50.0,5.0,2.0,3.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,1996.0,6.0,19.0,0.0,13.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C2,2587.0,10.0,15.0,0.0,23.0,5.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1888.0,8.0,8.0,0.0,11.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2224.0,8.0,27.0,0.0,9.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1041]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)

V1V3_tbl_matched_relative

,g__Cutibacterium|ASV1,g__Staphylococcus|ASV8,g__Lawsonella|ASV1,g__Streptococcus|ASV11,g__Cutibacterium|ASV4,g__Corynebacterium|ASV13,g__Streptococcus|ASV4,g__Micrococcus|ASV1,g__Veillonella|ASV3,g__Streptococcus|ASV6,...,g__Veillonella|ASV1,g__Staphylococcus|ASV4,g__Lactobacillus|ASV2,g__Streptococcus|ASV7,g__Corynebacterium|ASV8,g__Streptococcus|ASV1,g__Staphylococcus|ASV3,g__Corynebacterium|ASV4,g__Staphylococcus|ASV7,g__Staphylococcus|ASV2
LAMI.RD304.D14.C2,0.904469,0.047364,0.006155,0.010704,0.005084,0.006690,0.004014,0.000000,0.000535,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000268,0.000268,0.0
LAMI.RD305.D14.C1,0.974135,0.010077,0.002351,0.001680,0.000000,0.010413,0.000000,0.000000,0.000336,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0
LAMI.RD304.D4.C2,0.813947,0.142401,0.000799,0.000266,0.012510,0.006654,0.000000,0.002129,0.000000,0.000532,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0
LAMI.RD304.D16.C1,0.907255,0.032421,0.002126,0.001329,0.000266,0.026840,0.001063,0.000266,0.000266,0.001063,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000266,0.000000,0.0
LAMI.RD304.D7.C1,0.896598,0.060606,0.003721,0.001063,0.000532,0.013291,0.001329,0.000532,0.000797,0.000000,...,0.0,0.0,0.0,0.0,0.000266,0.00000,0.0,0.000532,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,0.978911,0.002943,0.009318,0.000000,0.006376,0.001471,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00049,0.0,0.000000,0.000000,0.0
LAMI.RD319.D21.C2,0.978072,0.003781,0.005671,0.000000,0.008696,0.001890,0.000000,0.000000,0.000756,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0
LAMI.RD319.D14.C1,0.982310,0.004162,0.004162,0.000000,0.005723,0.002601,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0
LAMI.RD319.D14.C2,0.942772,0.003391,0.011446,0.000000,0.003815,0.002120,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0


In [1042]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
7,LAMI.RD305.D23.C1,Acne_L
...,...,...
258,LAMI.RD305.D28.C1,Acne_L
259,LAMI.RD313.D4.C1,Acne_L
260,LAMI.RD305.D0.C2,Acne_L
261,LAMI.RD317.D21.C1,Acne_L


In [1043]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = f'../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [1044]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,"Pyroglutamic acid (mz: 130.0497016, RT: 0.6479..."
1,"(Iso)leucine (mz: 132.1017009, RT: 0.71511286)"
2,"Urocanic acid (mz: 139.0410336, RT: 0.6406374)"
3,"Urocanic acid (mz: 139.0587028, RT: 0.6315576)"
4,"Paracetamol (mz: 152.0704373, RT: 1.6221956)"
5,"Nicotine (mz: 163.1224157, RT: 0.634317)"
6,"Phenylalanine (mz: 166.086109, RT: 0.9777797)"
7,"Tyrosine (mz: 182.0809391, RT: 0.6720225)"
8,"Tryptophan (mz: 205.0969182, RT: 2.0760007)"
9,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)"


In [1045]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [1046]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.000,125692.330
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.3900,0.0000,0.000,0.000,0.00,0.000,10553.927
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.627,12473.092
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.586,40673.110
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.00,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.000,49464.902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.000,0.000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.000,220846.160
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.000,25695.014
LAMI.RD319.D2.C3,171846.72,106901.95,0.0000,1407.5230,25369.129,0.00,59064.336,833009.000,23485.404,2838344.20,...,25342.2100,0.0000,55986.290,0.0000,0.0000,0.000,0.000,0.00,0.000,11039.730


In [1047]:
# Drop the column from the DataFrame as it wasn't significant after univariate testing
# metaB_V4_tbl = metaB_V4_tbl.drop(columns=['C19H22O4 Linear diarylheptanoids'])

In [1048]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,0.265648,0.180123,0.002630,0.003544,0.002515,0.000000,0.086299,0.172743,0.050465,0.130990,...,0.005514,0.001103,0.054395,0.001492,0.000000,0.000000,0.000000,0.000000,0.000000,0.011177
LAMI.RD306.D9.C2,0.177092,0.195092,0.002975,0.000000,0.021844,0.015442,0.092514,0.308376,0.058006,0.026585,...,0.013238,0.001764,0.037799,0.001110,0.000000,0.000000,0.000000,0.000000,0.000000,0.000911
LAMI.RD304.D11.C1,0.269227,0.212329,0.007015,0.005441,0.002279,0.000000,0.088183,0.237725,0.048588,0.044292,...,0.008675,0.001392,0.011622,0.000000,0.000796,0.001662,0.003486,0.014728,0.004416,0.001847
LAMI.RD304.D11.C2,0.274761,0.172246,0.000844,0.000000,0.003662,0.000000,0.074110,0.211162,0.037169,0.046760,...,0.004916,0.000558,0.063903,0.003849,0.000656,0.003995,0.008503,0.038416,0.010752,0.009920
LAMI.RD304.D0.C1,0.172611,0.162410,0.000949,0.000698,0.009426,0.000000,0.061201,0.275662,0.028620,0.147184,...,0.009428,0.000000,0.089115,0.000673,0.000716,0.000000,0.000000,0.000000,0.000000,0.010318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.210388,0.224149,0.001544,0.001755,0.002638,0.000000,0.103823,0.247600,0.089449,0.042334,...,0.004404,0.000922,0.022046,0.010158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD308.D9.C3,0.139416,0.174247,0.000802,0.000780,0.004545,0.000000,0.071444,0.188158,0.066626,0.076921,...,0.006204,0.000000,0.016641,0.005532,0.000000,0.000000,0.000000,0.000000,0.000000,0.055971
LAMI.RD319.D2.C2,0.071895,0.064285,0.000478,0.000000,0.001260,0.000000,0.030511,0.142568,0.014543,0.645566,...,0.004206,0.000000,0.008744,0.002357,0.000000,0.000000,0.000000,0.000000,0.000000,0.003068
LAMI.RD319.D2.C3,0.040939,0.025467,0.000000,0.000335,0.006044,0.000000,0.014071,0.198447,0.005595,0.676175,...,0.006037,0.000000,0.013338,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002630


In [1049]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V4_tbl_relative.index = metaB_V4_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V4_tbl_relative.index)]

# Subset the DataFrame
metaB_V4_tbl_relative = metaB_V4_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V4_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20663/4148729606.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD310.D21.C1,0.161275,0.127148,0.003527,0.023439,0.003459,0.141652,0.055794,0.284922,0.031579,0.118801,...,0.005883,0.000000,0.002835,0.008138,0.000000,0.000000,0.000000,0.000000,0.000000,0.017880
LAMI.RD306.D18.C2,0.215800,0.170360,0.000851,0.002343,0.007131,0.001499,0.076037,0.219199,0.041103,0.185253,...,0.011764,0.001530,0.019971,0.002833,0.000430,0.000000,0.000000,0.008514,0.000000,0.007811
LAMI.RD306.D7.C2,0.234776,0.210658,0.008428,0.004283,0.003240,0.001876,0.089672,0.267062,0.048207,0.057453,...,0.008393,0.000511,0.009765,0.000453,0.000635,0.000000,0.000000,0.011246,0.000000,0.000000
LAMI.RD317.D14.C2,0.208521,0.147695,0.001010,0.001145,0.001750,0.019631,0.066396,0.136326,0.038252,0.280606,...,0.006773,0.001677,0.049383,0.003272,0.001062,0.000000,0.000000,0.000493,0.000000,0.000000
LAMI.RD304.D7.C1,0.262157,0.229848,0.008963,0.005448,0.002217,0.000304,0.105048,0.202240,0.050729,0.064113,...,0.005842,0.000000,0.005313,0.001395,0.001516,0.003591,0.009551,0.005548,0.009609,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD305.D28.C1,0.140246,0.131700,0.002039,0.000000,0.004383,0.000000,0.048748,0.233223,0.026580,0.294372,...,0.012756,0.000000,0.083917,0.002572,0.000000,0.000000,0.000000,0.000000,0.000000,0.001650
LAMI.RD313.D4.C1,0.234309,0.184645,0.001594,0.002245,0.011487,0.000000,0.088735,0.307286,0.044712,0.059806,...,0.004936,0.000000,0.012576,0.010067,0.000891,0.000000,0.000000,0.000000,0.000000,0.010636
LAMI.RD305.D0.C2,0.167853,0.102316,0.000621,0.003366,0.009479,0.000000,0.047204,0.283517,0.025330,0.281149,...,0.009113,0.000000,0.040475,0.002807,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD317.D21.C1,0.098352,0.080178,0.000415,0.000000,0.001394,0.015098,0.034518,0.100826,0.023286,0.576273,...,0.005320,0.000883,0.027064,0.002630,0.000000,0.000000,0.000000,0.006146,0.000000,0.006610


In [1050]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

# Strip everything before and including the last semicolon in column names
V4_tbl_matched.columns = V4_tbl_matched.columns.str.split(';').str[-1].str.strip()


# Instead of being fifth from last char, make the condition just IF there is a | overall
V4_tbl_matched.columns = [col if '|' in col else col + '|ASV1' for col in V4_tbl_matched.columns]

# Genus prefixes of interest
top_V4_features = ['g__Staphylococcus', 'g__Lawsonella', 'g__Streptococcus', 'g__Cutibacterium',
                   'g__Micrococcus', 'g__Lactobacillus', 'g__Rothia', 'g__Kocuria',
                   'g__Haemophilus', 'g__Corynebacterium', 'g__Veillonella']

# Find matching columns (those starting with any of the genus names)
matching_cols = [col for col in V4_tbl_matched.columns if col.startswith(tuple(top_V4_features))]

# Subset the dataframe to only those columns
V4_tbl_matched = V4_tbl_matched[matching_cols]


In [1051]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)

V4_tbl_matched_relative

,g__Staphylococcus|ASV3,g__Lawsonella|ASV1,g__Streptococcus|ASV5,g__Corynebacterium|ASV15,g__Corynebacterium|ASV9,g__Micrococcus|ASV1,g__Haemophilus|ASV1,g__Cutibacterium|ASV1,g__Lactobacillus|ASV1,g__Corynebacterium|ASV14,...,g__Rothia|ASV3,g__Corynebacterium|ASV5,g__Micrococcus|ASV2,g__Streptococcus|ASV7,g__Corynebacterium|ASV20,g__Lactobacillus|ASV4,g__Corynebacterium|ASV18,g__Staphylococcus|ASV4,g__Corynebacterium|ASV6,g__Staphylococcus|ASV1
LAMI.RD304.D0.C1,0.804363,0.007868,0.016452,0.045780,0.007153,0.009299,0.001788,0.005722,0.000715,0.011803,...,0.000715,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
LAMI.RD304.D0.C2,0.794069,0.018781,0.013509,0.055684,0.002965,0.006590,0.002965,0.003954,0.000659,0.002965,...,0.000000,0.0,0.0,0.0,0.003295,0.0,0.000329,0.0,0.0,0.0
LAMI.RD304.D18.C2,0.799801,0.023904,0.009296,0.050797,0.000332,0.029880,0.019256,0.013944,0.011952,0.002324,...,0.000664,0.0,0.0,0.0,0.004648,0.0,0.000000,0.0,0.0,0.0
LAMI.RD304.D9.C1,0.864402,0.013937,0.006678,0.017712,0.001161,0.006098,0.001452,0.003775,0.031359,0.002904,...,0.000000,0.0,0.0,0.0,0.000290,0.0,0.003194,0.0,0.0,0.0
LAMI.RD304.D21.C2,0.520056,0.015834,0.090429,0.066502,0.000352,0.017945,0.005278,0.010204,0.146024,0.003167,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.003167,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,0.459459,0.364865,0.006757,0.054054,0.040541,0.027027,0.000000,0.006757,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
LAMI.RD319.D28.C1,0.215385,0.492308,0.010256,0.056410,0.092308,0.000000,0.046154,0.000000,0.000000,0.010256,...,0.010256,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
LAMI.RD318.D9.C2,0.612335,0.101322,0.017621,0.079295,0.008811,0.052863,0.008811,0.044053,0.000000,0.022026,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
LAMI.RD319.D28.C2,0.412698,0.428571,0.000000,0.055556,0.015873,0.015873,0.007937,0.000000,0.007937,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0


In [1052]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
8,LAMI.RD304.D7.C1,Acne_L
...,...,...
258,LAMI.RD305.D28.C1,Acne_L
259,LAMI.RD313.D4.C1,Acne_L
260,LAMI.RD305.D0.C2,Acne_L
261,LAMI.RD317.D21.C1,Acne_L


In [1053]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [1054]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

In [1055]:
V1V3_tbl_matched

,g__Cutibacterium|ASV1,g__Staphylococcus|ASV8,g__Lawsonella|ASV1,g__Streptococcus|ASV11,g__Cutibacterium|ASV4,g__Corynebacterium|ASV13,g__Streptococcus|ASV4,g__Micrococcus|ASV1,g__Veillonella|ASV3,g__Streptococcus|ASV6,...,g__Veillonella|ASV1,g__Staphylococcus|ASV4,g__Lactobacillus|ASV2,g__Streptococcus|ASV7,g__Corynebacterium|ASV8,g__Streptococcus|ASV1,g__Staphylococcus|ASV3,g__Corynebacterium|ASV4,g__Staphylococcus|ASV7,g__Staphylococcus|ASV2
LAMI.RD304.D14.C2,3380.0,177.0,23.0,40.0,19.0,25.0,15.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
LAMI.RD305.D14.C1,2900.0,30.0,7.0,5.0,0.0,31.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D4.C2,3058.0,535.0,3.0,1.0,47.0,25.0,0.0,8.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D16.C1,3414.0,122.0,8.0,5.0,1.0,101.0,4.0,1.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
LAMI.RD304.D7.C1,3373.0,228.0,14.0,4.0,2.0,50.0,5.0,2.0,3.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,1996.0,6.0,19.0,0.0,13.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C2,2587.0,10.0,15.0,0.0,23.0,5.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1888.0,8.0,8.0,0.0,11.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2224.0,8.0,27.0,0.0,9.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Combine V1-V3 and V4 tables

In [1056]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Merge by taking average of values row-wise
merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# Sort columns by total sum (descending order)
merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

merged_tbl

,g__Cutibacterium|ASV1,g__Staphylococcus|ASV3,g__Lawsonella|ASV1,g__Streptococcus|ASV5,g__Corynebacterium|ASV15,g__Staphylococcus|ASV8,g__Corynebacterium|ASV9,g__Haemophilus|ASV1,g__Lactobacillus|ASV1,g__Micrococcus|ASV1,...,g__Staphylococcus|ASV6,g__Streptococcus|ASV9,g__Corynebacterium|ASV12,g__Staphylococcus|ASV5,g__Streptococcus|ASV8,g__Streptococcus|ASV12,g__Streptococcus|ASV10,g__Staphylococcus|ASV4,g__Corynebacterium|ASV5,g__Staphylococcus|ASV7
LAMI.RD304.D14.C2,3382.0,1859.0,284.0,266.0,199.0,177.0,21.0,47.0,134.0,17.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
LAMI.RD305.D14.C1,2905.0,226.0,60.0,9.0,120.0,30.0,0.0,14.0,0.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D4.C2,3073.0,2951.0,21.0,28.0,86.0,535.0,17.0,7.0,0.0,43.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D16.C1,3433.0,1460.0,94.0,78.0,625.0,122.0,0.0,16.0,645.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D7.C1,3384.0,2238.0,117.0,43.0,224.0,228.0,0.0,16.0,110.0,55.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,1997.0,42.0,39.0,1.0,6.0,6.0,6.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C2,2588.0,57.0,38.0,0.0,5.0,10.0,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1889.0,28.0,14.0,0.0,3.0,8.0,1.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2224.0,72.0,60.0,1.0,18.0,8.0,0.0,1.0,0.0,1.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1057]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium|ASV1,g__Staphylococcus|ASV3,g__Lawsonella|ASV1,g__Streptococcus|ASV5,g__Corynebacterium|ASV15,g__Corynebacterium|ASV9,g__Haemophilus|ASV1,g__Micrococcus|ASV1,g__Lactobacillus|ASV1,g__Lactobacillus|ASV3,...,g__Staphylococcus|ASV7,g__Staphylococcus|ASV8,g__Staphylococcus|ASV9,g__Streptococcus|ASV10,g__Streptococcus|ASV11,g__Streptococcus|ASV12,g__Streptococcus|ASV13,g__Streptococcus|ASV8,g__Streptococcus|ASV9,g__Veillonella|ASV4
LAMI.RD304.D14.C2,1691.0,929.5,142.0,133.0,99.5,10.5,23.5,8.5,67.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD305.D14.C1,1452.5,113.0,30.0,4.5,60.0,0.0,7.0,3.5,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD304.D4.C2,1536.5,1475.5,10.5,14.0,43.0,8.5,3.5,21.5,0.0,143.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD304.D16.C1,1716.5,730.0,47.0,39.0,312.5,0.0,8.0,10.0,322.5,6.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD304.D7.C1,1692.0,1119.0,58.5,21.5,112.0,0.0,8.0,27.5,55.0,133.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,998.5,21.0,19.5,0.5,3.0,3.0,0.0,1.5,0.0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD319.D21.C2,1294.0,28.5,19.0,0.0,2.5,1.0,0.0,0.5,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD319.D14.C1,944.5,14.0,7.0,0.0,1.5,0.5,0.0,1.0,0.0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LAMI.RD319.D14.C2,1112.0,36.0,30.0,0.5,9.0,0.0,0.5,0.5,0.0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1058]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)

# Remove columns with NaN values
merged_tbl_relative = merged_tbl_relative.dropna(axis=1, how='any')

merged_tbl_relative

,g__Cutibacterium|ASV1,g__Staphylococcus|ASV3,g__Lawsonella|ASV1,g__Streptococcus|ASV5,g__Corynebacterium|ASV15,g__Corynebacterium|ASV9,g__Haemophilus|ASV1,g__Micrococcus|ASV1,g__Lactobacillus|ASV1,g__Lactobacillus|ASV3,...,g__Corynebacterium|ASV6,g__Rothia|ASV2,g__Corynebacterium|ASV8,g__Streptococcus|ASV1,g__Staphylococcus|ASV1,g__Streptococcus|ASV7,g__Staphylococcus|ASV2,g__Corynebacterium|ASV12,g__Staphylococcus|ASV4,g__Corynebacterium|ASV5
LAMI.RD304.D14.C2,0.524666,0.288396,0.044058,0.041266,0.030872,0.003258,0.007291,0.002637,0.020788,0.000776,...,0.000000,0.000465,0.000000,0.000000,0.0,0.002172,0.0,0.000000,0.0,0.0
LAMI.RD305.D14.C1,0.858705,0.066805,0.017736,0.002660,0.035471,0.000000,0.004138,0.002069,0.000000,0.000000,...,0.000000,0.000296,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
LAMI.RD304.D4.C2,0.466667,0.448140,0.003189,0.004252,0.013060,0.002582,0.001063,0.006530,0.000000,0.043432,...,0.000152,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
LAMI.RD304.D16.C1,0.515775,0.219351,0.014123,0.011719,0.093900,0.000000,0.002404,0.003005,0.096905,0.001953,...,0.000150,0.000000,0.000000,0.000000,0.0,0.000150,0.0,0.000000,0.0,0.0
LAMI.RD304.D7.C1,0.505527,0.334329,0.017478,0.006424,0.033463,0.000000,0.002390,0.008216,0.016433,0.039886,...,0.001046,0.000000,0.000598,0.000000,0.0,0.000000,0.0,0.000149,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,0.947343,0.019924,0.018501,0.000474,0.002846,0.002846,0.000000,0.001423,0.000000,0.000474,...,0.000000,0.000000,0.000474,0.000474,0.0,0.000000,0.0,0.000000,0.0,0.0
LAMI.RD319.D21.C2,0.957455,0.021088,0.014058,0.000000,0.001850,0.000740,0.000000,0.000370,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
LAMI.RD319.D14.C1,0.970211,0.014381,0.007191,0.000000,0.001541,0.000514,0.000000,0.001027,0.000000,0.000514,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
LAMI.RD319.D14.C2,0.928601,0.030063,0.025052,0.000418,0.007516,0.000000,0.000418,0.000418,0.000000,0.000418,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000835,0.0,0.0


In [1059]:
merged_tbl_relative.columns

Index(['g__Cutibacterium|ASV1', 'g__Staphylococcus|ASV3', 'g__Lawsonella|ASV1',
       'g__Streptococcus|ASV5', 'g__Corynebacterium|ASV15',
       'g__Corynebacterium|ASV9', 'g__Haemophilus|ASV1', 'g__Micrococcus|ASV1',
       'g__Lactobacillus|ASV1', 'g__Lactobacillus|ASV3',
       'g__Streptococcus|ASV2', 'g__Corynebacterium|ASV14',
       'g__Corynebacterium|ASV13', 'g__Veillonella|ASV3',
       'g__Corynebacterium|ASV10', 'g__Veillonella|ASV2',
       'g__Corynebacterium|ASV1', 'g__Corynebacterium|ASV16',
       'g__Corynebacterium|ASV2', 'g__Streptococcus|ASV4',
       'g__Corynebacterium|ASV3', 'g__Corynebacterium|ASV11',
       'g__Corynebacterium|ASV7', 'g__Veillonella|ASV1',
       'g__Lactobacillus|ASV2', 'g__Streptococcus|ASV3',
       'g__Corynebacterium|ASV4', 'g__Streptococcus|ASV6', 'g__Rothia|ASV1',
       'g__Kocuria|ASV1', 'g__Corynebacterium|ASV6', 'g__Rothia|ASV2',
       'g__Corynebacterium|ASV8', 'g__Streptococcus|ASV1',
       'g__Staphylococcus|ASV1', 'g__Strept

In [1060]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = f'../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [1061]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
metaB_MERGED_tbl.index.name = None

In [1062]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]
metaB_MERGED_tbl

,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","Cyclo(his-pro) (mz: 235.1186308, RT: 0.6402779)","Phenylbenzimidazole sulfonic acid (mz: 275.0481837, RT: 2.761023)","Nicotine (mz: 163.1224157, RT: 0.634317)",...,"Uridine (mz: 245.0765944, RT: 0.65536463)","Nobiletin (mz: 403.138254, RT: 5.258404)","xylometazoline (mz: 245.2008949, RT: 4.3418465)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)"
LAMI.RD306.D9.C2,308019.97,3572847.500,2051791.50,2260333.20,1071868.20,672054.40,437938.200,472807.600,0.000,178911.470,...,74657.664,0.000,0.0,12865.3900,0.000,34466.4840,0.000,0.0000,0.0000,20433.4900
LAMI.RD304.D11.C1,299140.06,1605551.000,1818308.60,1434033.20,595568.94,328151.12,78491.530,218959.970,0.000,0.000,...,26177.530,23546.951,0.0,0.0000,29824.627,47379.2730,11225.194,36750.4000,5375.8525,9403.6520
LAMI.RD304.D11.C2,191713.67,865753.500,1126503.20,706197.75,303845.30,152392.72,261996.800,119362.910,0.000,0.000,...,19284.926,34863.656,0.0,15780.3090,44084.586,3460.0974,16378.752,0.0000,2691.0256,2286.2397
LAMI.RD304.D0.C1,705570.80,1321474.400,827466.30,778566.10,293386.34,137201.39,427202.060,132853.970,0.000,0.000,...,15696.905,0.000,0.0,3228.1143,0.000,4547.2640,0.000,3345.8184,3431.0034,0.0000
LAMI.RD306.D11.C2,227752.42,1761918.500,1711784.50,1283604.40,642067.25,415650.38,259815.220,171644.750,0.000,15334.904,...,31315.730,0.000,0.0,22948.7460,0.000,8437.5560,0.000,9085.7370,14211.7630,11457.0290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD310.D0.C2,273832.88,1569170.100,1726083.60,1394403.50,569740.44,376634.80,502334.800,154011.110,3511.621,0.000,...,20015.640,0.000,0.0,55764.8440,0.000,10637.6280,0.000,7607.8213,0.0000,0.0000
LAMI.RD304.D9.C1,192508.81,1254550.800,1284439.80,908266.90,457527.78,219415.83,1562542.800,179900.100,0.000,6508.036,...,20449.324,372729.840,0.0,0.0000,347685.400,8513.0610,133879.400,0.0000,9194.5180,3996.3018
LAMI.RD304.D9.C2,252395.50,1474280.400,1476603.80,1116758.50,521042.84,267440.38,251054.770,153787.600,0.000,9236.615,...,34124.523,1293640.800,0.0,5651.8830,1256496.500,21766.3980,471657.100,0.0000,12874.5510,4033.7710
LAMI.RD319.D2.C2,5406879.00,1194061.900,602147.10,538412.94,255542.72,121803.89,73231.560,62017.848,9720.738,0.000,...,16367.659,0.000,0.0,19741.2460,0.000,4006.6484,0.000,0.0000,0.0000,0.0000


In [1063]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","Cyclo(his-pro) (mz: 235.1186308, RT: 0.6402779)","Phenylbenzimidazole sulfonic acid (mz: 275.0481837, RT: 2.761023)","Nicotine (mz: 163.1224157, RT: 0.634317)",...,"Uridine (mz: 245.0765944, RT: 0.65536463)","Nobiletin (mz: 403.138254, RT: 5.258404)","xylometazoline (mz: 245.2008949, RT: 4.3418465)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)"
LAMI.RD306.D9.C2,0.026585,0.308376,0.177092,0.195092,0.092514,0.058006,0.037799,0.040808,0.000000,0.015442,...,0.006444,0.000000,0.0,0.001110,0.000000,0.002975,0.000000,0.000000,0.000000,0.001764
LAMI.RD304.D11.C1,0.044292,0.237725,0.269227,0.212329,0.088183,0.048588,0.011622,0.032420,0.000000,0.000000,...,0.003876,0.003486,0.0,0.000000,0.004416,0.007015,0.001662,0.005441,0.000796,0.001392
LAMI.RD304.D11.C2,0.046760,0.211162,0.274761,0.172246,0.074110,0.037169,0.063903,0.029113,0.000000,0.000000,...,0.004704,0.008503,0.0,0.003849,0.010752,0.000844,0.003995,0.000000,0.000656,0.000558
LAMI.RD304.D0.C1,0.147184,0.275662,0.172611,0.162410,0.061201,0.028620,0.089115,0.027714,0.000000,0.000000,...,0.003274,0.000000,0.0,0.000673,0.000000,0.000949,0.000000,0.000698,0.000716,0.000000
LAMI.RD306.D11.C2,0.033317,0.257742,0.250408,0.187772,0.093925,0.060803,0.038007,0.025109,0.000000,0.002243,...,0.004581,0.000000,0.0,0.003357,0.000000,0.001234,0.000000,0.001329,0.002079,0.001676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD310.D0.C2,0.040750,0.233512,0.256863,0.207505,0.084785,0.056048,0.074754,0.022919,0.000523,0.000000,...,0.002979,0.000000,0.0,0.008299,0.000000,0.001583,0.000000,0.001132,0.000000,0.000000
LAMI.RD304.D9.C1,0.027426,0.178733,0.182991,0.129398,0.065183,0.031260,0.222611,0.025630,0.000000,0.000927,...,0.002913,0.053102,0.0,0.000000,0.049534,0.001213,0.019073,0.000000,0.001310,0.000569
LAMI.RD304.D9.C2,0.029085,0.169888,0.170156,0.128689,0.060042,0.030818,0.028930,0.017722,0.000000,0.001064,...,0.003932,0.149072,0.0,0.000651,0.144792,0.002508,0.054351,0.000000,0.001484,0.000465
LAMI.RD319.D2.C2,0.645566,0.142568,0.071895,0.064285,0.030511,0.014543,0.008744,0.007405,0.001161,0.000000,...,0.001954,0.000000,0.0,0.002357,0.000000,0.000478,0.000000,0.000000,0.000000,0.000000


In [1064]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = f'../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative_{skin_group}.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)